# Text Preprocessing for Sentiment Analysis

In [12]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import vertex_ai_utils as vai

# Display settings
pd.set_option('display.max_colwidth', 150)

In [13]:
# Load dataset
df = vai.load_twitter_data('data/Tweets.csv')
print(f"\nDataset shape: {df.shape}")
df.head()

Successfully loaded 14640 tweets from data/Tweets.csv

Dataset shape: (14640, 15)


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials to the experience... tacky.,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I need to take another trip!,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse",NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing about it,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [14]:
# Example tweet cleaning
sample_tweet = df['text'].iloc[0]
print(f"  {vai.clean_text(sample_tweet)}")

  what said.


In [15]:
# Show multiple examples
for i in range(5):
    original = df['text'].iloc[i]
    cleaned = vai.clean_text(original)
    print(f"\n{i+1}. Original: {original}")
    print(f"   Cleaned:  {cleaned}")


1. Original: @VirginAmerica What @dhepburn said.
   Cleaned:  what said.

2. Original: @VirginAmerica plus you've added commercials to the experience... tacky.
   Cleaned:  plus you've added commercials to the experience... tacky.

3. Original: @VirginAmerica I didn't today... Must mean I need to take another trip!
   Cleaned:  i didn't today... must mean i need to take another trip!

4. Original: @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse
   Cleaned:  it's really aggressive to blast obnoxious "entertainment" in your guests' faces and they have little recourse

5. Original: @VirginAmerica and it's a really big bad thing about it
   Cleaned:  and it's a really big bad thing about it


In [16]:
# Preprocess a single tweet with full pipeline
sample = "@airline Your service is NOT good! Check http://example.com #disappointed"

print("Original:")
print(f"  {sample}")
print("\nPreprocessed:")
print(f"  {vai.preprocess_tweet(sample)}")

Original:
  @airline Your service is NOT good! Check http://example.com #disappointed

Preprocessed:
  service not good check disappointed


In [17]:
# Apply preprocessing to all tweets
df_processed = vai.preprocess_dataframe(
    df,
    text_column='text',
    output_column='text_processed',
    remove_stopwords_flag=True,
    lemmatize=True,
    keep_negation=True
)

print("\nPreprocessed dataset shape:", df_processed.shape)

Preprocessing 14640 tweets...
Average original length: 103.8 chars
Average processed length: 59.3 chars
Length reduction: 42.9%

Preprocessed dataset shape: (14640, 16)


In [18]:
# View original vs processed
df_processed[['text', 'text_processed', 'airline_sentiment']].head(10)

,text,text_processed,airline_sentiment
0,@VirginAmerica What @dhepburn said.,said,neutral
1,@VirginAmerica plus you've added commercials to the experience... tacky.,plus youve added commercial experience tacky,positive
2,@VirginAmerica I didn't today... Must mean I need to take another trip!,didnt today must mean need take another trip,neutral
3,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse",really aggressive blast obnoxious entertainment guest face little recourse,negative
4,@VirginAmerica and it's a really big bad thing about it,really big bad thing,negative
5,@VirginAmerica seriously would pay $30 a flight for seats that didn't have this playing.\nit's really the only bad thing about flying VA,seriously would pay 30 flight seat didnt playing really bad thing flying va,negative
6,"@VirginAmerica yes, nearly every time I fly VX this “ear worm” won’t go away :)",yes nearly every time fly vx “ ear worm ” ’ go away,positive
7,"@VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there. https://t.co/mWpG7grEZP",really missed prime opportunity men without hat parody,neutral
8,"@virginamerica Well, I didn't…but NOW I DO! :-D",well didnt…but,positive
9,"@VirginAmerica it was amazing, and arrived an hour early. You're too good to me.",amazing arrived hour early youre good,positive


## 4. Analyze Preprocessing Impact

In [19]:
# Comprehensive impact analysis
vai.analyze_preprocessing_impact(df_processed, 'text', 'text_processed')


 Text Length Statistics:
   Original:
  Mean: 103.8 chars
  Median: 114.0 chars
   Processed:
  Mean: 59.3 chars
  Median: 63.0 chars
   Reduction: 42.9%

 Word Count Statistics:
   Original:
  Mean: 17.7 words
  Median: 19.0 words
   Processed:
  Mean: 9.5 words
  Median: 10.0 words
   Reduction: 46.3%

Example 1:
  Original:  @SouthwestAir you're my early frontrunner for best airline! #oscars2016...
  Processed: youre early frontrunner best airline oscars2016...

Example 2:
  Original:  @USAirways how is it that my flt to EWR was Cancelled Flightled yet flts to NYC from USAirways are s...
  Processed: flt ewr cancelled flightled yet flts nyc usairways still flying...

Example 3:
  Original:  @JetBlue what is going on with your BDL to DCA flights yesterday and today?! Why is every single one...
  Processed: going bdl dca flight yesterday today every single one getting delayed...

Example 4:
  Original:  @JetBlue do they have to depart from Washington, D.C.??...
  Processed: depart wa

In [20]:
# Show examples by sentiment
for sentiment in ['positive', 'neutral', 'negative']:
    print(f"{sentiment.upper()} EXAMPLES")

    sample_df = df_processed[df_processed['airline_sentiment'] == sentiment].head(3)
    
    for idx, (_, row) in enumerate(sample_df.iterrows(), 1):
        print(f"\n{idx}. Original:  {row['text'][:100]}...")
        print(f"   Processed: {row['text_processed'][:100]}...")

POSITIVE EXAMPLES

1. Original:  @VirginAmerica plus you've added commercials to the experience... tacky....
   Processed: plus youve added commercial experience tacky...

2. Original:  @VirginAmerica yes, nearly every time I fly VX this “ear worm” won’t go away :)...
   Processed: yes nearly every time fly vx “ ear worm ” ’ go away...

3. Original:  @virginamerica Well, I didn't…but NOW I DO! :-D...
   Processed: well didnt…but...
NEUTRAL EXAMPLES

1. Original:  @VirginAmerica What @dhepburn said....
   Processed: said...

2. Original:  @VirginAmerica I didn't today... Must mean I need to take another trip!...
   Processed: didnt today must mean need take another trip...

3. Original:  @VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there. https://t.co/mW...
   Processed: really missed prime opportunity men without hat parody...
NEGATIVE EXAMPLES

1. Original:  @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces 

In [21]:
# Save processed dataset
output_path = 'data/processed/tweets_preprocessed.csv'
df_processed.to_csv(output_path, index=False)
print(f"Preprocessed data saved to: {output_path}")

# Also save train/val/test splits with preprocessing
train_df, val_df, test_df = vai.split_train_val_test(
    df_processed,
    stratify_column='airline_sentiment'
)

train_df.to_csv('data/processed/train_preprocessed.csv', index=False)
val_df.to_csv('data/processed/val_preprocessed.csv', index=False)
test_df.to_csv('data/processed/test_preprocessed.csv', index=False)

print("\nTrain/val/test splits saved with preprocessing!")

Preprocessed data saved to: data/processed/tweets_preprocessed.csv
Training set:   10,248 records (70.0%)
Validation set:  2,196 records (15.0%)
Test set:        2,196 records (15.0%)

  Sentiment distribution verification:
   Train: positive: 16.15%  neutral: 21.17%  negative: 62.69%     Val  : positive: 16.12%  neutral: 21.17%  negative: 62.70%     Test : positive: 16.12%  neutral: 21.17%  negative: 62.70%  
Train/val/test splits saved with preprocessing!


In [22]:
# Create JSONL files with preprocessed text
vai.prepare_data_for_vertex_ai(
    train_df,
    text_column='text_processed',  # Use preprocessed text
    label_column='airline_sentiment',
    output_path='data/processed/train_preprocessed.jsonl'
)

vai.prepare_data_for_vertex_ai(
    val_df,
    text_column='text_processed',
    label_column='airline_sentiment',
    output_path='data/processed/val_preprocessed.jsonl'
)

vai.prepare_data_for_vertex_ai(
    test_df,
    text_column='text_processed',
    label_column='airline_sentiment',
    output_path='data/processed/test_preprocessed.jsonl'
)

print("\n JSONL files created with preprocessed text!")

Saved to: data/processed/train_preprocessed.jsonl
Saved to: data/processed/val_preprocessed.jsonl
Saved to: data/processed/test_preprocessed.jsonl

 JSONL files created with preprocessed text!
